In [1]:
import pandas as pd
import numpy as np

In [2]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = "supermarine45/4be-dataset"
repo_files = list_repo_files(REPO_ID, repo_type="dataset")


def load_csv_group(prefixes):
    csv_files = sorted(
        file_name
        for file_name in repo_files
        if file_name.endswith(".csv") and any(file_name.startswith(prefix) for prefix in prefixes)
    )

    frames = []
    for file_name in csv_files:
        local_path = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=file_name
        )
        frames.append(pd.read_csv(local_path, dtype=str, low_memory=False))

    return pd.concat(frames, ignore_index=True, sort=False)


def load_split_for_schema(schema_root, split_name):
    prefixes = [
        f"{schema_root}/attack/{split_name}/",
        f"{schema_root}/normal/{split_name}/",
    ]
    return load_csv_group(prefixes)


reduced_schema_roots = [
    "Option1/option1_nf_unsw_dos_as_ddos_reduced_schema",
    "Option2/option2_nf_unsw_base_cse_native_ddos_reduced_schema",
]

full_schema_roots = [
    "Option1/option1_nf_unsw_dos_as_ddos",
    "Option2/option2_nf_unsw_base_cse_native_ddos",
]


def load_schema_splits(schema_roots):
    return {
        split_name: pd.concat(
            [load_split_for_schema(schema_root, split_name) for schema_root in schema_roots],
            ignore_index=True,
            sort=False,
        )
        for split_name in ["train", "test", "validation"]
    }


reduced_splits = load_schema_splits(reduced_schema_roots)
full_splits = load_schema_splits(full_schema_roots)

# Keep the existing downstream notebook cells working on the training split.
df_reduced = reduced_splits["train"].copy()
df_full = full_splits["train"].copy()

# Make the held-out splits available for later evaluation.
df_reduced_test = reduced_splits["test"].copy()
df_reduced_validation = reduced_splits["validation"].copy()
df_full_test = full_splits["test"].copy()
df_full_validation = full_splits["validation"].copy()

# Convert reduced-schema numeric columns so the later statistics cells keep working.
reduced_numeric_columns = [
    'src_port', 'dst_port', 'protocol', 'outbound_byte_ratio', 'duration',
    'packets_per_second', 'bytes_per_second', 'inter_packet_arrival_mean',
    'inter_packet_arrival_std', 'total_packets', 'total_bytes',
    'packet_size_avg', 'packet_size_std', 'dataset_id', 'row_in_window',
    'is_seeded_ddos', 'burst_id'
]

for frame in [df_reduced, df_reduced_test, df_reduced_validation]:
    for column in reduced_numeric_columns:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')

print(f"df_reduced train shape: {df_reduced.shape}")
print(f"df_reduced test shape: {df_reduced_test.shape}")
print(f"df_reduced validation shape: {df_reduced_validation.shape}")
print(f"df_full train shape: {df_full.shape}")
print(f"df_full test shape: {df_full_test.shape}")
print(f"df_full validation shape: {df_full_validation.shape}")

/opt/miniconda3/envs/fp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


df_reduced train shape: (3200000, 25)
df_reduced test shape: (800000, 25)
df_reduced validation shape: (800000, 25)
df_full train shape: (3200000, 64)
df_full test shape: (800000, 64)
df_full validation shape: (800000, 64)


In [3]:
df_full.iloc[1000]

FLOW_START_MILLISECONDS                      9000
FLOW_END_MILLISECONDS                       10005
IPV4_SRC_ADDR                          59.166.0.8
L4_SRC_PORT                                 12295
IPV4_DST_ADDR                       149.171.126.4
                                    ...          
is_seeded_ddos                                  0
burst_id                                      NaN
burst_phase                                   NaN
bot_pool_size                                  59
source_dataset             NF-UNSW-NB15-v3-Benign
Name: 1000, Length: 64, dtype: str

# PRELIMINARY DATA ANALYSIS

In [4]:
print(f"df_reduced shape: {df_reduced.shape}")
print(f"df_full shape: {df_full.shape}")

df_reduced shape: (3200000, 25)
df_full shape: (3200000, 64)


In [5]:
print("df_reduced columns:")
print(df_reduced.columns.tolist())
print("\ndf_full columns:")
print(df_full.columns.tolist())

df_reduced columns:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'outbound_byte_ratio', 'duration', 'packets_per_second', 'bytes_per_second', 'inter_packet_arrival_mean', 'inter_packet_arrival_std', 'total_packets', 'total_bytes', 'packet_size_avg', 'packet_size_std', 'Label', 'Attack', 'scenario', 'split', 'dataset_id', 'row_in_window', 'is_seeded_ddos', 'burst_id', 'burst_phase', 'source_dataset']

df_full columns:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO

In [6]:
# COLUMN Statistics

# Analyze Attack column distribution
attack_dist = df_full['Attack'].value_counts(normalize=True) * 100
attack_counts = df_full['Attack'].value_counts()

print('=' * 80)
print('ATTACK COLUMN DISTRIBUTION - PERCENTAGE AND COUNTS')
print('=' * 80)
print('\nPercentage breakdown:')
print(attack_dist.sort_values(ascending=False))

print('\nAbsolute counts:')
print(attack_counts.sort_values(ascending=False))

print('\nSummary:')
benign_pct = (df_full['Attack'] == 'Benign').sum() / len(df_full) * 100
attack_pct = 100 - benign_pct
print(f'Benign: {benign_pct:.2f}% ({(df_full["Attack"] == "Benign").sum():,} rows)')
print(f'DDoS/Attack: {attack_pct:.2f}% ({(df_full["Attack"] != "Benign").sum():,} rows)')
print(f'Total rows: {len(df_full):,}')

ATTACK COLUMN DISTRIBUTION - PERCENTAGE AND COUNTS

Percentage breakdown:
Attack
Benign    99.034687
DDoS       0.965313
Name: proportion, dtype: float64

Absolute counts:
Attack
Benign    3169110
DDoS        30890
Name: count, dtype: int64

Summary:
Benign: 99.03% (3,169,110 rows)
DDoS/Attack: 0.97% (30,890 rows)
Total rows: 3,200,000


In [7]:
print('=' * 80)
print('COMPREHENSIVE DATA STATISTICS')
print('=' * 80)

# 1. DATASET SHAPE AND SIZE
print('\n1. DATASET SHAPE AND SIZE')
print('-' * 80)
print(f'Total rows: {len(df_reduced):,}')
print(f'Total columns: {len(df_reduced.columns):,}')
print(f'Memory usage: {df_reduced.memory_usage(deep=True).sum() / 1024**3:.2f} GB')
print(f'Approx rows per MB: {len(df_reduced) / (df_reduced.memory_usage(deep=True).sum() / 1024**2):.0f}')

# 2. MISSING VALUES
print('\n2. MISSING VALUES')
print('-' * 80)
missing = df_reduced.isnull().sum()
missing_pct = (missing / len(df_reduced) * 100).round(2)
missing_summary = pd.DataFrame({'Missing_Count': missing, 'Percentage': missing_pct})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
if len(missing_summary) > 0:
    print(missing_summary)
else:
    print('No missing values found!')

# 3. DATA TYPES
print('\n3. DATA TYPES DISTRIBUTION')
print('-' * 80)
dtype_counts = df_reduced.dtypes.value_counts()
print(dtype_counts)

# 4. NUMERIC COLUMN STATISTICS
print('\n4. NUMERIC COLUMNS - SUMMARY STATISTICS')
print('-' * 80)
numeric_stats = df_reduced.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
print(numeric_stats[['min', '25%', '50%', '75%', '95%', '99%', 'max', 'std']].round(2))

# 5. ATTACK DISTRIBUTION BY SCENARIO
print('\n5. ATTACK DISTRIBUTION BY SCENARIO')
print('-' * 80)
scenario_attack = pd.crosstab(df_reduced['scenario'], df_reduced['Attack'], margins=True)
print(scenario_attack)

# 6. ATTACK DISTRIBUTION BY DATASET
print('\n6. ATTACK DISTRIBUTION BY DATASET ID')
print('-' * 80)
dataset_attack = pd.crosstab(df_reduced['dataset_id'], df_reduced['Attack'], margins=True)
print(dataset_attack)

# 7. SOURCE IP STATISTICS
print('\n7. SOURCE IP STATISTICS')
print('-' * 80)
n_unique_src_ips = df_reduced['src_ip'].nunique()
src_ip_flow_counts = df_reduced['src_ip'].value_counts()
print(f'Unique source IPs: {n_unique_src_ips:,}')
print(f'Mean flows per source: {src_ip_flow_counts.mean():.1f}')
print(f'Median flows per source: {src_ip_flow_counts.median():.1f}')
print(f'Max flows from single source: {src_ip_flow_counts.max():,}')
print(f'Min flows per source: {src_ip_flow_counts.min()}')
print(f'95th percentile flows: {src_ip_flow_counts.quantile(0.95):.0f}')
print('\nTop 10 most active source IPs:')
print(src_ip_flow_counts.head(10))

# 8. DESTINATION IP/PORT STATISTICS
print('\n8. DESTINATION IP AND PORT STATISTICS')
print('-' * 80)
n_unique_dst_ips = df_reduced['dst_ip'].nunique()
n_unique_dst_ports = df_reduced['dst_port'].nunique()
print(f'Unique destination IPs: {n_unique_dst_ips:,}')
print(f'Unique destination ports: {n_unique_dst_ports:,}')
print(f'Port range: {df_reduced["dst_port"].min()} - {df_reduced["dst_port"].max()}')
print('\nTop 10 most targeted ports:')
print(df_reduced['dst_port'].value_counts().head(10))

# 9. PROTOCOL DISTRIBUTION
print('\n9. PROTOCOL DISTRIBUTION')
print('-' * 80)
proto_counts = df_reduced['protocol'].value_counts()
proto_pct = (proto_counts / len(df_reduced) * 100).round(2)
proto_summary = pd.DataFrame({'Count': proto_counts, 'Percentage': proto_pct})
print(proto_summary)

# 10. ATTACK PATTERN BY PROTOCOL
print('\n10. ATTACK PATTERN BY PROTOCOL')
print('-' * 80)
protocol_attack = pd.crosstab(df_reduced['protocol'], df_reduced['Attack'], margins=True)
print(protocol_attack)

# 11. FLOW DURATION STATISTICS
print('\n11. FLOW DURATION STATISTICS')
print('-' * 80)
print(f'Mean duration: {df_reduced["duration"].mean():.4f} sec')
print(f'Median duration: {df_reduced["duration"].median():.4f} sec')
print(f'Min duration: {df_reduced["duration"].min():.4f} sec')
print(f'Max duration: {df_reduced["duration"].max():.4f} sec')
print(f'Std dev: {df_reduced["duration"].std():.4f} sec')

# 12. TRAFFIC RATE STATISTICS (packets_per_second, bytes_per_second)
print('\n12. TRAFFIC RATE STATISTICS')
print('-' * 80)
print('\nPackets per second:')
print(f'  Mean: {df_reduced["packets_per_second"].mean():.2f}')
print(f'  Median: {df_reduced["packets_per_second"].median():.2f}')
print(f'  95th percentile: {df_reduced["packets_per_second"].quantile(0.95):.2f}')
print(f'  99th percentile: {df_reduced["packets_per_second"].quantile(0.99):.2f}')
print(f'  Max: {df_reduced["packets_per_second"].max():.2f}')

print('\nBytes per second:')
print(f'  Mean: {df_reduced["bytes_per_second"].mean():.2f}')
print(f'  Median: {df_reduced["bytes_per_second"].median():.2f}')
print(f'  95th percentile: {df_reduced["bytes_per_second"].quantile(0.95):.2f}')
print(f'  99th percentile: {df_reduced["bytes_per_second"].quantile(0.99):.2f}')
print(f'  Max: {df_reduced["bytes_per_second"].max():.2f}')

# 13. TRAFFIC VOLUME STATISTICS
print('\n13. TRAFFIC VOLUME STATISTICS')
print('-' * 80)
print(f'Mean total packets per flow: {df_reduced["total_packets"].mean():.2f}')
print(f'Median total packets per flow: {df_reduced["total_packets"].median():.2f}')
print(f'Max total packets in single flow: {df_reduced["total_packets"].max():,}')
print(f'Mean total bytes per flow: {df_reduced["total_bytes"].mean():.2f}')
print(f'Median total bytes per flow: {df_reduced["total_bytes"].median():.2f}')
print(f'Max total bytes in single flow: {df_reduced["total_bytes"].max():,}')

# 14. PACKET SIZE ANALYSIS
print('\n14. PACKET SIZE ANALYSIS')
print('-' * 80)
print(f'Mean packet size: {df_reduced["packet_size_avg"].mean():.2f} bytes')
print(f'Median packet size: {df_reduced["packet_size_avg"].median():.2f} bytes')
print(f'Min packet size: {df_reduced["packet_size_avg"].min():.2f} bytes')
print(f'Max packet size: {df_reduced["packet_size_avg"].max():.2f} bytes')
print(f'Std dev packet size: {df_reduced["packet_size_std"].mean():.2f} bytes')

# 15. OUTBOUND BYTE RATIO ANALYSIS
print('\n15. OUTBOUND BYTE RATIO ANALYSIS (Asymmetry)')
print('-' * 80)
print(f'Mean outbound ratio: {df_reduced["outbound_byte_ratio"].mean():.4f}')
print(f'Median outbound ratio: {df_reduced["outbound_byte_ratio"].median():.4f}')
print(f'Min ratio: {df_reduced["outbound_byte_ratio"].min():.4f}')
print(f'Max ratio: {df_reduced["outbound_byte_ratio"].max():.4f}')
print(f'Flows with ratio < 0.1 (inbound heavy): {(df_reduced["outbound_byte_ratio"] < 0.1).sum():,} ({(df_reduced["outbound_byte_ratio"] < 0.1).mean()*100:.2f}%)')

# 16. ATTACK vs BENIGN COMPARISON
print('\n16. ATTACK vs BENIGN - KEY METRICS COMPARISON')
print('-' * 80)
attack_benign_compare = df_reduced.groupby('Attack')[['packets_per_second', 'bytes_per_second', 'duration', 'total_packets', 'total_bytes', 'outbound_byte_ratio']].agg(['mean', 'median', 'std', 'min', 'max'])
print(attack_benign_compare.round(2))

# 17. BURSTS AND SEEDING INFO
print('\n17. BURST AND SEEDING INFORMATION')
print('-' * 80)
print(f'Flows with seeded DDoS flag: {df_reduced["is_seeded_ddos"].sum():,} ({df_reduced["is_seeded_ddos"].mean()*100:.2f}%)')
print(f'Unique burst IDs: {df_reduced["burst_id"].nunique():,}')
print(f'Burst phases: {df_reduced["burst_phase"].nunique() if "burst_phase" in df_reduced.columns else "N/A"}')

# 18. DATASET SPLITS
print('\n18. TRAIN/TEST SPLIT')
print('-' * 80)
split_dist = df_reduced['split'].value_counts()
split_pct = (split_dist / len(df_reduced) * 100).round(2)
split_summary = pd.DataFrame({'Count': split_dist, 'Percentage': split_pct})
print(split_summary)

COMPREHENSIVE DATA STATISTICS

1. DATASET SHAPE AND SIZE
--------------------------------------------------------------------------------
Total rows: 3,200,000
Total columns: 25
Memory usage: 0.78 GB
Approx rows per MB: 3984

2. MISSING VALUES
--------------------------------------------------------------------------------
             Missing_Count  Percentage
protocol           3198173       99.94
burst_id           3169110       99.03
burst_phase        3169110       99.03

3. DATA TYPES DISTRIBUTION
--------------------------------------------------------------------------------
float64    10
str         8
int64       7
Name: count, dtype: int64

4. NUMERIC COLUMNS - SUMMARY STATISTICS
--------------------------------------------------------------------------------
                             min       25%        50%        75%         95%  \
src_port                    0.00  16145.00   32841.00   49154.00    62281.00   
dst_port                    0.00     25.00      80.00   1648

In [8]:
# Drop columns with missing values
df_reduced = df_reduced.drop(columns=['burst_id', 'burst_phase'], errors='ignore')

PRELIMINARY CONCLUSION: Class distribution uneven, skewed towards the 0 class

# FEATURE SELECTION

In [9]:
# Feature policy for src_ip-window aggregated model
# Never use audit/leakage fields as model inputs.
audit_fields = [
    'Attack', 'Label', 'scenario', 'split', 'dataset_id',
    'burst_id', 'burst_phase', 'is_seeded_ddos', 'source_dataset', 'row_in_window'
]

# src_ip is the grouping key for aggregation, not a model input.
identity_fields = ['src_ip', 'dst_ip']

# Requested feature family mapped to available reduced-schema columns.
selected_feature_candidates = [
    'dst_port',                    # destination port behavior proxy
    'protocol',                    # protocol behavior
    'packets_per_second',          # packet rate
    'bytes_per_second',            # byte rate
    'duration',                    # duration
    'total_packets',               # traffic volume
    'total_bytes',                 # traffic volume
    'packet_size_avg',             # packet-size stats
    'packet_size_std',             # packet-size stats
    'outbound_byte_ratio',         # outbound ratio
    'inter_packet_arrival_mean',   # temporal behavior
    'inter_packet_arrival_std'     # temporal behavior
]

available_selected_features = [
    col for col in selected_feature_candidates if col in df_reduced.columns
]
missing_selected_features = [
    col for col in selected_feature_candidates if col not in df_reduced.columns
]

print('Selected features found:', available_selected_features)
if missing_selected_features:
    print('Selected features missing in df_reduced:', missing_selected_features)

# Build modeling frame from selected features only.
X_selected = df_reduced[available_selected_features].copy()

# Encode categorical selected features only (protocol is categorical behaviorally).
categorical_selected = [col for col in ['protocol'] if col in X_selected.columns]
if categorical_selected:
    X_selected = pd.get_dummies(X_selected, columns=categorical_selected, drop_first=True, dtype=float)

# Ensure numeric matrix and clean invalid rows.
X_selected = X_selected.apply(pd.to_numeric, errors='coerce')
X_selected = X_selected.astype(float)

# Binary target: Attack vs Benign
y_selected = (df_reduced['Attack'].astype(str).str.lower() != 'benign').astype(int)

mask_clean = ~(
    X_selected.isna().any(axis=1)
    | np.isinf(X_selected).any(axis=1)
    | y_selected.isna()
)

X_clean = X_selected.loc[mask_clean].reset_index(drop=True)
y_clean = y_selected.loc[mask_clean].reset_index(drop=True)

print(f'X_clean shape: {X_clean.shape}')
print('Class balance (0=Benign, 1=Attack):')
print(y_clean.value_counts(normalize=True).sort_index())

Selected features found: ['dst_port', 'protocol', 'packets_per_second', 'bytes_per_second', 'duration', 'total_packets', 'total_bytes', 'packet_size_avg', 'packet_size_std', 'outbound_byte_ratio', 'inter_packet_arrival_mean', 'inter_packet_arrival_std']
X_clean shape: (3200000, 14)
Class balance (0=Benign, 1=Attack):
Attack
0    0.990347
1    0.009653
Name: proportion, dtype: float64


In [10]:
# Calculate VIF for selected features to check for multicollinearity.
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF on selected feature set only
sample_size = min(10000, len(X_clean))
X_sample = X_clean.sample(n=sample_size, random_state=42)
X_vif = sm.add_constant(X_sample, has_constant='add')

print('Calculating VIF on selected features...')

vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

vif_data = (
    vif_data[vif_data['Feature'] != 'const']
    .sort_values(by='VIF', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 80)
print('VIF ON SELECTED FEATURES')
print('=' * 80)
print(vif_data.head(20))

high_vif_features = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
good_features = vif_data[vif_data['VIF'] <= 10]['Feature'].tolist()

print(f'\nHigh VIF features (>10): {len(high_vif_features)}')
print(high_vif_features[:20])
print(f'\nGood features (<=10): {len(good_features)}')
print(good_features[:20])

Calculating VIF on selected features...

VIF ON SELECTED FEATURES
                      Feature        VIF
0    inter_packet_arrival_std  34.892259
1   inter_packet_arrival_mean  33.941018
2             packet_size_avg  21.328092
3                 total_bytes  20.652510
4             packet_size_std  16.502933
5               total_packets  13.521166
6            bytes_per_second   4.290966
7                    duration   3.301020
8          packets_per_second   3.220302
9               protocol_89.0   2.707476
10                   dst_port   1.579728
11        outbound_byte_ratio   1.403400
12             protocol_132.0        NaN
13             protocol_211.0        NaN

High VIF features (>10): 6
['inter_packet_arrival_std', 'inter_packet_arrival_mean', 'packet_size_avg', 'total_bytes', 'packet_size_std', 'total_packets']

Good features (<=10): 6
['bytes_per_second', 'duration', 'packets_per_second', 'protocol_89.0', 'dst_port', 'outbound_byte_ratio']


/opt/miniconda3/envs/fp/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


# OLS to check significance of features

In [11]:
import statsmodels.api as sm

# OLS on selected feature set only
X_ols = sm.add_constant(X_clean, has_constant='add')
ols_selected = sm.OLS(y_clean, X_ols).fit()

print('\n' + '=' * 80)
print('OLS RESULTS - SELECTED FEATURE SET')
print('=' * 80)
print(ols_selected.summary())


OLS RESULTS - SELECTED FEATURE SET
                            OLS Regression Results                            
Dep. Variable:                 Attack   R-squared:                       0.083
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                 2.081e+04
Date:                Sat, 09 May 2026   Prob (F-statistic):               0.00
Time:                        22:51:56   Log-Likelihood:             3.0391e+06
No. Observations:             3200000   AIC:                        -6.078e+06
Df Residuals:                 3199985   BIC:                        -6.078e+06
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------

In [12]:
# Significant features from OLS using selected feature set
print('\n' + '=' * 80)
print('SIGNIFICANT FEATURES (p < 0.05) - SELECTED SET')
print('=' * 80)

sig_features_selected = []
for feature in ols_selected.params.index:
    if feature == 'const':
        continue
    pvalue = ols_selected.pvalues[feature]
    if pvalue < 0.05:
        coef = ols_selected.params[feature]
        sig_features_selected.append({
            'feature': feature,
            'coefficient': coef,
            'p_value': pvalue,
            'abs_coef': abs(coef)
        })

sig_features_selected.sort(key=lambda x: x['abs_coef'], reverse=True)
print(f'Found {len(sig_features_selected)} significant features')
for i, feat in enumerate(sig_features_selected[:20], 1):
    print(f"{i:2d}. {feat['feature']:35s} | coef: {feat['coefficient']:12.6f} | p-value: {feat['p_value']:.2e}")

# Cross-check OLS significance with multicollinearity filter
sig_feature_names = {x['feature'] for x in sig_features_selected}
vif_ok_set = set(good_features)
final_candidate_features = sorted(sig_feature_names.intersection(vif_ok_set))

print('\n' + '=' * 80)
print('FINAL CANDIDATE FEATURES (OLS significant AND VIF <= 10)')
print('=' * 80)
print(final_candidate_features)
print(f'Total final candidate features: {len(final_candidate_features)}')

print('\nDropped by policy (audit/leakage fields):')
print(audit_fields + identity_fields)


SIGNIFICANT FEATURES (p < 0.05) - SELECTED SET
Found 14 significant features
 1. protocol_132.0                      | coef:     0.988791 | p-value: 2.36e-220
 2. protocol_211.0                      | coef:     0.982364 | p-value: 7.92e-50
 3. protocol_89.0                       | coef:    -0.073969 | p-value: 9.19e-61
 4. outbound_byte_ratio                 | coef:    -0.021843 | p-value: 0.00e+00
 5. inter_packet_arrival_mean           | coef:     0.000441 | p-value: 0.00e+00
 6. duration                            | coef:     0.000441 | p-value: 5.52e-40
 7. inter_packet_arrival_std            | coef:    -0.000085 | p-value: 0.00e+00
 8. packet_size_avg                     | coef:    -0.000078 | p-value: 0.00e+00
 9. packet_size_std                     | coef:     0.000054 | p-value: 0.00e+00
10. total_packets                       | coef:     0.000004 | p-value: 1.04e-213
11. packets_per_second                  | coef:    -0.000002 | p-value: 0.00e+00
12. dst_port                 

In [13]:
print('final_candidate_features:', final_candidate_features)
print('num_final_features:', len(final_candidate_features))

final_candidate_features: ['bytes_per_second', 'dst_port', 'duration', 'outbound_byte_ratio', 'packets_per_second', 'protocol_89.0']
num_final_features: 6


In [14]:
"""
def engineer_ddos_features(df):
    # Define thresholds based on common DDoS characteristics
    SMALL_PACKET_THRESHOLD = 128
    HIGH_PPS_THRESHOLD = 1000  
    LOW_OUTBOUND_THRESHOLD = 0.1 

    # 1. Base Aggregations
    # We use a dictionary for standard mean/max/sum operations
    agg_dict = {
        'dst_ip': 'nunique',
        'dst_port': 'nunique',
        'protocol': 'nunique',
        'packets_per_second': ['mean', 'max'],
        'bytes_per_second': ['mean', 'max'],
        'duration': ['mean', 'max'],
        'total_packets': 'sum',
        'total_bytes': 'sum',
        'packet_size_avg': ['mean', 'std'],
        'outbound_byte_ratio': ['mean', 'min'],
        'Label': 'max' # If any flow in the window is an attack, the aggregate is 1
    }

    # Group by Source IP
    # Note: If your window spans multiple bursts, you might group by ['src_ip', 'dataset_id']
    grouped = df.groupby('src_ip')
    
    # Execute standard aggregations
    features = grouped.agg(agg_dict)
    
    # Flatten MultiIndex columns (e.g., ('duration', 'mean') -> 'duration_mean')
    features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in features.columns]
    
    # 2. Custom "Share" and "Concentration" Features
    # Concentration: Ratio of flows going to the most frequent destination
    features['concentration_dst_ip'] = grouped['dst_ip'].apply(
        lambda x: x.value_counts().iloc[0] / len(x) if not x.empty else 0
    )
    features['concentration_dst_port'] = grouped['dst_port'].apply(
        lambda x: x.value_counts().iloc[0] / len(x) if not x.empty else 0
    )

    # Protocol Shares (TCP=6, UDP=17, ICMP=1)
    features['share_tcp'] = grouped['protocol'].apply(lambda x: (x == 6).mean())
    features['share_udp'] = grouped['protocol'].apply(lambda x: (x == 17).mean())
    features['share_icmp'] = grouped['protocol'].apply(lambda x: (x == 1).mean())

    # Behavioral Shares
    features['share_small_packets'] = grouped['packet_size_avg'].apply(
        lambda x: (x < SMALL_PACKET_THRESHOLD).mean()
    )
    features['share_high_pps'] = grouped['packets_per_second'].apply(
        lambda x: (x > HIGH_PPS_THRESHOLD).mean()
    )
    features['share_low_outbound'] = grouped['outbound_byte_ratio'].apply(
        lambda x: (x < LOW_OUTBOUND_THRESHOLD).mean()
    )
    
    # Number of flows generated by source
    features['num_flows'] = grouped.size()

    # Final cleanup: Replace NaNs from std() calculations with 0
    return features.fillna(0).reset_index()

# Integration into your notebook:
df_engineered = engineer_ddos_features(df_reduced)
display(df_engineered)
"""

'\ndef engineer_ddos_features(df):\n    # Define thresholds based on common DDoS characteristics\n    SMALL_PACKET_THRESHOLD = 128\n    HIGH_PPS_THRESHOLD = 1000  \n    LOW_OUTBOUND_THRESHOLD = 0.1 \n\n    # 1. Base Aggregations\n    # We use a dictionary for standard mean/max/sum operations\n    agg_dict = {\n        \'dst_ip\': \'nunique\',\n        \'dst_port\': \'nunique\',\n        \'protocol\': \'nunique\',\n        \'packets_per_second\': [\'mean\', \'max\'],\n        \'bytes_per_second\': [\'mean\', \'max\'],\n        \'duration\': [\'mean\', \'max\'],\n        \'total_packets\': \'sum\',\n        \'total_bytes\': \'sum\',\n        \'packet_size_avg\': [\'mean\', \'std\'],\n        \'outbound_byte_ratio\': [\'mean\', \'min\'],\n        \'Label\': \'max\' # If any flow in the window is an attack, the aggregate is 1\n    }\n\n    # Group by Source IP\n    # Note: If your window spans multiple bursts, you might group by [\'src_ip\', \'dataset_id\']\n    grouped = df.groupby(\'src_

Final features

1. bytes_per_second: Measures the volume of data flow over time.

2. dst_port: Acts as a proxy for destination behavior (e.g., targeting specific services).

3. duration: The length of the network flow.

4. outbound_byte_ratio: A critical indicator of asymmetry, which is highly significant in DDoS detection.

5. packets_per_second: Measures the intensity/rate of the packet transmission.

6. aggregated features (mean, max, and sum)

7. concentration_dst_ip, concentration_dst_port

8. TCP, UDP, or ICMP\

9. share_small_packets: Identifying "noisy" small-packet floods.

10. share_low_outbound

Dropped features:

1. High Multicollinearity (Dropped due to VIF > 10): 
total_bytes, total_packets, packet_size_avg, packet_size_std, and inter_packet_arrival 

2. Audit Fields: Label, scenario, split, dataset_id, burst_id, burst_phase, and is_seeded_ddos

## Lag and Rolling Window Features
The data is not a classic evenly sampled time series, but it does contain flow order within each `src_ip` and dataset window. That makes it suitable for lag and rolling-history features that capture short-term bursts before a DDoS label appears.

In [15]:
"""
def add_lag_rolling_features_per_flow(df_input, X_clean, y_clean):
    """
    Add lag and rolling features to X_clean at the PER-FLOW level (not aggregated).
    Captures temporal patterns in individual flows from each source IP.
    
    For each feature, creates:
    - lag1: value from previous flow
    - roll3_mean: rolling mean of previous 3 flows  
    - delta: change from previous flow
    - spike: whether current > 1.5x rolling mean
    """
    X_with_lags = X_clean.copy()
    
    # Combine src_ip with features (for grouping)
    df_temp = df_input.copy().reset_index(drop=True)
    X_temp = X_clean.copy().reset_index(drop=True)
    
    df_with_src = pd.concat([
        df_temp[['src_ip']].iloc[:len(X_temp)].reset_index(drop=True), 
        X_temp.reset_index(drop=True)
    ], axis=1)
    
    # Define features to lag (skip dummy-encoded categoricals)
    lag_features = [
        col for col in X_clean.columns 
        if col not in ['protocol_6', 'protocol_17', 'protocol_1']
        and X_clean[col].dtype in ['float64', 'int64']
    ]
    
    def add_lags_per_source(group_df):
        """Add lag/rolling features within each source IP group"""
        for col in lag_features:
            if col in group_df.columns:
                # Lag-1: previous flow from same source
                group_df[f'{col}_lag1'] = group_df[col].shift(1)
                
                # Rolling-3: mean of previous 3 flows
                group_df[f'{col}_roll3_mean'] = group_df[col].shift(1).rolling(window=3, min_periods=1).mean()
                
                # Delta: change from previous flow
                group_df[f'{col}_delta'] = group_df[col] - group_df[f'{col}_lag1']
                
                # Spike: is current > 1.5x rolling mean? (burst indicator)
                group_df[f'{col}_spike'] = (group_df[col] > group_df[f'{col}_roll3_mean'] * 1.5).astype(float)
        
        return group_df
    
    # Group by src_ip and apply lags
    X_with_lags = df_with_src.groupby('src_ip', group_keys=False).apply(add_lags_per_source)
    X_with_lags = X_with_lags.drop(columns=['src_ip']).reset_index(drop=True)
    
    # Fill NaN lags with 0 (first flow from each source has no previous)
    X_with_lags = X_with_lags.fillna(0)
    
    return X_with_lags, y_clean

# === APPLY LAG/ROLLING FEATURES AT PER-FLOW LEVEL ===
print('Adding lag and rolling features to per-flow data (X_clean, y_clean)...')
X_lag, y_lag = add_lag_rolling_features_per_flow(df_reduced, X_clean, y_clean)

print(f'Original X_clean shape: {X_clean.shape}')
print(f'X_lag (with rolling features) shape: {X_lag.shape}')
print(f'New lag/rolling features added: {X_lag.shape[1] - X_clean.shape[1]}')
print(f'\nNew columns (sample):')
new_cols = [c for c in X_lag.columns if c not in X_clean.columns]
print(new_cols[:10])
"""

IndentationError: unexpected indent (3411871167.py, line 4)

In [ ]:
from pathlib import Path

output_path = Path('/Users/fagunawan/Documents/quantum_hack/code/4BE/QCentroid - ETH Hackathon 2026/df_reduced.csv')
df_reduced.to_csv(output_path, index=False)

print(f'Saved df_reduced to: {output_path}')
print(f'Rows: {len(df_reduced):,}')
print(f'Columns: {len(df_reduced.columns):,}')

Saved df_reduced to: /Users/fagunawan/Documents/quantum_hack/code/4BE/QCentroid - ETH Hackathon 2026/df_reduced.csv
Rows: 3,200,000
Columns: 25


In [ ]:
# === EXPORT df_reduced WITH FINAL CANDIDATE FEATURES ===
# Select only the final_candidate_features columns that exist in df_reduced
# (some may be dummy-encoded from X_clean)
available_final_features = [col for col in final_candidate_features if col in df_reduced.columns]
missing_final_features = [col for col in final_candidate_features if col not in df_reduced.columns]

if missing_final_features:
    print(f'Note: {len(missing_final_features)} features not in df_reduced (likely dummy-encoded): {missing_final_features[:5]}')

df_final_features = df_reduced[available_final_features].copy()

print(f'df_final_features shape: {df_final_features.shape}')
print(f'Columns: {list(df_final_features.columns)}')
print(f'\nFirst few rows:')
display(df_final_features.head(10))
print(f'\nData types:')
print(df_final_features.dtypes)
print(f'\nBasic statistics:')
display(df_final_features.describe())

Note: 1 features not in df_reduced (likely dummy-encoded): ['protocol_89.0']
df_final_features shape: (3200000, 5)
Columns: ['bytes_per_second', 'dst_port', 'duration', 'outbound_byte_ratio', 'packets_per_second']

First few rows:


,bytes_per_second,dst_port,duration,outbound_byte_ratio,packets_per_second
0,3.751250e+05,21,0.008,0.583139,6000.000000
1,4.818871e+05,38306,0.124,0.934565,1064.516129
2,2.872471e+05,8344,0.085,0.895888,964.705882
3,4.730435e+04,48047,0.046,0.852941,304.347826
4,1.267500e+04,21,0.280,0.579318,200.000000
5,2.180000e+05,111,0.004,0.348624,2000.000000
6,1.502815e+06,25,0.027,0.083300,3481.481481
7,2.032500e+06,62335,0.012,0.891513,6833.333333
8,1.768889e+05,6881,0.018,0.516332,1888.888889
9,3.650000e+05,21,0.002,0.642466,6000.000000



Data types:
bytes_per_second       float64
dst_port                 int64
duration               float64
outbound_byte_ratio    float64
packets_per_second     float64
dtype: object

Basic statistics:


,bytes_per_second,dst_port,duration,outbound_byte_ratio,packets_per_second
count,3.200000e+06,3.200000e+06,3.200000e+06,3.200000e+06,3.200000e+06
mean,5.122227e+05,1.155191e+04,5.815147e-01,6.390905e-01,2.745759e+03
std,6.397487e+05,1.861447e+04,3.606154e+00,2.473818e-01,2.662717e+03
min,4.894249e+00,0.000000e+00,1.000000e-03,0.000000e+00,6.250000e-02
25%,2.101098e+04,2.500000e+01,4.000000e-03,5.493830e-01,1.634859e+02
50%,2.920000e+05,8.000000e+01,2.800000e-02,5.846000e-01,2.000000e+03
75%,7.861818e+05,1.648000e+04,3.490000e-01,8.884310e-01,4.000000e+03
max,9.024000e+06,6.553500e+04,1.209940e+02,9.955560e-01,2.000000e+04
